<a href="https://colab.research.google.com/github/csikasote/igc-mu-cibemba/blob/main/generate_dataset_splits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Clone dataset from Github

In [2]:
!git clone https://github.com/csikasote/igc-mu-cibemba.git

Cloning into 'igc-mu-cibemba'...
remote: Enumerating objects: 109038, done.
remote: Counting objects: 100% (8000/8000), done.
remote: Compressing objects: 100% (7960/7960), done.
remote: Total 109038 (delta 80), reused 7906 (delta 36), pack-reused 101038
Receiving objects: 100% (109038/109038), 17.81 GiB | 30.36 MiB/s, done.
Resolving deltas: 100% (208/208), done.
Checking out files: 100% (108359/108359), done.


In [1]:
import pandas as pd
import numpy as np
import json
import os

In [3]:
absolute_data_path = "/content/igc-mu-cibemba/data"

In [4]:
def get_speech_task_split(abs_path, dst_path, df=None, split_name=None, task=None):
    if task == 'asr':
        new_df = df[['audio_id','bem_transcription']].copy()
    elif task == 'st':
        source_lang = 'bem_transcription'
        target_lang = 'en_translation'
        new_df = df[['audio_id',source_lang, target_lang]].copy()
        
    new_df['path'] = abs_path + '/audio/' + new_df['audio_id']
    new_df = new_df.dropna(subset=['path'])
    new_df = new_df.drop(columns=['audio_id'])
    if task == 'asr':
        new_df = new_df.rename(columns={'bem_transcription':'sentence', 'path':'audio'})
    elif task == 'st':
        new_df = new_df.rename(columns={source_lang:'bemba', target_lang:'english','path':'audio'})
    new_df.to_csv(f'{dst_path}/{split_name}.csv', sep='\t' , header=0, index=None)
    print(f'No. of {split_name} records: {len(new_df)}')

In [5]:
def get_mt_task_split(dst_path, df=None, split_name=None, task=None):
        source_lang = 'bem_transcription'
        target_lang = 'en_translation'
        
        # source and target for test set
        new_df_bemba = df[[source_lang]].copy()
        new_df_english = df[[target_lang]].copy()

        # save as separate files
        new_df_bemba.to_csv(f'{dst_path}/{split_name}.bem', sep='\t' , header=0, index=None)
        print(f'No. of {split_name} records [bem]: {len(new_df_bemba)}')
        new_df_english.to_csv(f'{dst_path}/{split_name}.en', sep='\t' , header=0, index=None)
        print(f'No. of {split_name} records [en]: {len(new_df_english)}')


In [18]:
def generate_dataset_splits(src_path, dst_path=None, task=None):
    # load the splits
    test_df = pd.read_json(f"{src_path}/splits/test.jsonl", orient='records', lines=True)
    valid_df = pd.read_json(f"{src_path}/splits/valid.jsonl", orient='records', lines=True)
    train_df = pd.read_json(f"{src_path}/splits/train.jsonl", orient='records', lines=True)
    
    if task=='asr':
        dst_path = os.path.abspath(src_path) + f"/splits/{task}"
        if not os.path.exists(dst_path):
            os.makedirs(dst_path)
        print('ASR dataset splits:\n')
        get_speech_task_split(src_path, dst_path, test_df, 'test', task)
        get_speech_task_split(src_path, dst_path, valid_df, 'valid', task)
        get_speech_task_split(src_path, dst_path, train_df, 'train', task)
                
    elif task=="mt":
        dst_path = os.path.abspath(src_path) + f"/splits/{task}"
        if not os.path.exists(dst_path):
            os.makedirs(dst_path)
        
        print('MT dataset splits:\n')
        get_mt_task_split(dst_path, test_df, 'test', task)
        get_mt_task_split(dst_path, valid_df, 'valid', task)
        get_mt_task_split(dst_path, train_df, 'train', task)

    elif task=='st':
        dst_path = os.path.abspath(src_path) + f"/splits/{task}"
        if not os.path.exists(dst_path):
            os.makedirs(dst_path)
        print('ST dataset splits:\n')   
        get_speech_task_split(src_path, dst_path, test_df, 'test', task)
        get_speech_task_split(src_path, dst_path, valid_df, 'valid', task)
        get_speech_task_split(src_path, dst_path, train_df, 'train', task)       
    
    return print(f'\nSuccessfully generated splits in: {dst_path}')

In [19]:
# specify the task name for which you want to generate the dataset splits for
generate_dataset_splits(absolute_data_path, task='mt')

MT dataset splits:

No. of test records [bem]: 2779
No. of test records [en]: 2779
No. of valid records [bem]: 2782
No. of valid records [en]: 2782
No. of train records [bem]: 82375
No. of train records [en]: 82375

Successfully generated splits in: /content/igc-mu-cibemba/data/splits/mt
